# 🗓 Calendar Agent — Colab LoRA Training

Qwen2.5-0.5B에 LoRA 어댑터를 부착해서 합성 데이터(JSONL)로 파인튜닝.

## 실행 전 준비
1. **Runtime > Change runtime type > T4 GPU**
2. **GitHub PAT** (repo scope) — https://github.com/settings/tokens/new?type=classic
3. **데이터 파일 3개** — Drive `MyDrive/calendar-agent-data/`에 미리 업로드 (권장) 또는 노트북에 직접 업로드 다이얼로그:
   - `v1_train.jsonl`
   - `v1_val.jsonl`
   - `golden.jsonl`

## 실행 순서
셀을 위에서 아래로 순서대로 실행. Colab 세션이 끊겨 재시작하면 처음부터 다시 실행 (clone, install 포함).

예상 시간: 데이터 업로드 ~1분 + 학습 ~1시간 + 다운로드 ~1분.

## 0. GPU 확인

`Tesla T4` 또는 16GB+ VRAM GPU 표시되어야 함. "No GPU"면 런타임 설정 다시.

In [ ]:
!nvidia-smi

## 1. 레포 clone (Private)

처음이면 PAT 입력해 clone, 이미 있으면 `git pull`로 최신화.

In [ ]:
import os, getpass

if not os.path.exists('calendar-agent'):
    token = getpass.getpass('GitHub PAT (repo scope): ').strip()
    !git clone https://{token}@github.com/soo-vibe/calendar-agent.git
    token = None
else:
    print('calendar-agent already cloned, pulling latest…')
    !cd calendar-agent && git pull

%cd calendar-agent
!git log --oneline -1

## 2. 학습 의존성 설치 + Colab 호환 cleanup

한 셀에서 둘 다:
1. `pip install -e .[train]` — transformers/peft/trl/accelerate/bitsandbytes/datasets/wandb
2. `pip uninstall torchao -y` — Colab 기본의 `torchao 0.10`이 현재 `peft`와 충돌. 제거해도 LoRA엔 영향 없음.

2~3분 소요. dependency 경고는 무시 OK.

In [ ]:
!pip install -q -e .[train]
!pip uninstall torchao -y
import torch
print(f'\ntorch={torch.__version__}  cuda={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

## 3. 데이터 업로드 (스마트: 이미 있으면 skip)

필요 파일:
- `data/processed/v1_train.jsonl`
- `data/processed/v1_val.jsonl`
- `data/eval/golden.jsonl`

**자동 처리 순서**:
1. 이미 다 있으면 skip
2. Drive에 `MyDrive/calendar-agent-data/` 있으면 마운트해서 복사
3. 둘 다 실패하면 직접 업로드 다이얼로그 (3개 파일 한 번에 Ctrl+클릭 선택)

In [ ]:
import shutil
from pathlib import Path

TRAIN = Path('data/processed/v1_train.jsonl')
VAL   = Path('data/processed/v1_val.jsonl')
GOLD  = Path('data/eval/golden.jsonl')
TRAIN.parent.mkdir(parents=True, exist_ok=True)
GOLD.parent.mkdir(parents=True, exist_ok=True)

def all_present():
    return TRAIN.exists() and VAL.exists() and GOLD.exists()

if all_present():
    print('데이터 파일 모두 존재 — skip')
else:
    # 1) Drive 시도
    drive_src = Path('/content/drive/MyDrive/calendar-agent-data')
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        if drive_src.exists():
            for fn, dst in [('v1_train.jsonl', TRAIN), ('v1_val.jsonl', VAL), ('golden.jsonl', GOLD)]:
                src = drive_src / fn
                if src.exists() and not dst.exists():
                    shutil.copy(src, dst)
                    print(f'Drive → {dst}')
    except Exception as e:
        print(f'Drive 마운트 실패 ({type(e).__name__}) — 직접 업로드로 fallback')

    # 2) 직접 업로드 (남은 파일만)
    if not all_present():
        missing = [p.name for p in [TRAIN, VAL, GOLD] if not p.exists()]
        print(f'\n부족한 파일: {missing}')
        print('아래에서 3개 파일을 한 번에 선택하세요 (Ctrl+클릭).')
        from google.colab import files
        uploaded = files.upload()
        for fn in uploaded:
            if 'train' in fn and not TRAIN.exists():
                shutil.move(fn, TRAIN)
            elif 'val' in fn and not VAL.exists():
                shutil.move(fn, VAL)
            elif 'golden' in fn and not GOLD.exists():
                shutil.move(fn, GOLD)

print()
for p in [TRAIN, VAL, GOLD]:
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'{status}  {p}')

## 4. Sanity check + config 자동 조정

1. 3개 파일 존재·shape 검증
2. WandB 끄기 (Colab 첫 사용 시 인증 흐름 회피)
3. T4 같이 bf16 미지원이면 자동 fp16 전환

In [ ]:
import orjson, torch
from pathlib import Path

for p in [Path('data/processed/v1_train.jsonl'), Path('data/processed/v1_val.jsonl'), Path('data/eval/golden.jsonl')]:
    assert p.exists(), f'MISSING: {p}'
    n = sum(1 for line in open(p, 'rb') if line.strip())
    print(f'{p}: {n} records')

# wandb off
!sed -i 's/report_to: wandb/report_to: none/' configs/train.yaml

# bf16 -> fp16 if needed
if not torch.cuda.is_bf16_supported():
    !sed -i 's/bf16: true/bf16: false/' configs/train.yaml
    !sed -i 's/fp16: false/fp16: true/' configs/train.yaml
    !sed -i "s|bnb_4bit_compute_dtype: bfloat16|bnb_4bit_compute_dtype: float16|" configs/train.yaml
    print('GPU에 bf16 미지원 — fp16로 전환')
else:
    print('bf16 사용')

!grep -E 'report_to|bf16|fp16|bnb_4bit_compute' configs/train.yaml

## 5. 학습 실행

1820건 × 3 epochs ≈ 156 step.  T4 기준 ~1시간 (step당 20~30초).

**진행 패턴**:
- 모델 다운로드 (첫 회 ~5초, 이후 캐시)
- 토크나이즈 (~2초)
- 학습 루프: 20 step마다 `{'loss': X.X, ...}` 출력
- epoch 끝마다 `{'eval_loss': X.X, ...}` 출력
- 끝나면 `[train] 완료 → models/lora/v1`

**Colab Free idle disconnect 약 90분** — 브라우저 탭 활성 유지.

In [ ]:
!python scripts/train_lora.py --config configs/train.yaml

## 6. 결과 패키징 + 다운로드 (+ Drive 백업)

LoRA 어댑터(~34MB) + 학습 설정 zip → 로컬 + Drive 둘 다 저장.
이어서 로컬에서 `merge_lora.py` → `quantize.sh` → 추론 테스트.

In [ ]:
!ls -la models/lora/v1/
!zip -r lora_v1.zip models/lora/v1 configs/train.yaml configs/lora.yaml configs/model_qwen.yaml

# Drive에도 백업 (분실 방지)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    !cp lora_v1.zip /content/drive/MyDrive/
    print('Drive 백업 완료: MyDrive/lora_v1.zip')
except Exception as e:
    print(f'Drive 백업 skip ({type(e).__name__})')

from google.colab import files
files.download('lora_v1.zip')